# Quickstart: Ridge Mapping

**Use case:** You have a vector database embedded with an "old" model and want to use a "new" model. isotrieve learns a linear mapping between the two spaces so you can transform existing vectors without re-embedding your entire corpus.

**When you'd reach for this:** Any time you want to migrate to a better embedding model but re-embedding is expensive or slow.

**What you need installed:** `isotrieve`, `numpy`, `scikit-learn` (all included in the base install).

**Estimated runtime:** ~1 minute.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krish1925/AECP/blob/main/isotrieve-python/notebooks/01_quickstart_ridge_mapping.ipynb)

In [ ]:
# Install isotrieve (no-op if already installed)
!pip install -q isotrieve numpy scikit-learn

In [ ]:
import numpy as np

from isotrieve import RidgeMapping
from isotrieve.quality.gate import QualityGate

print("isotrieve imported successfully")

## 1. Generate synthetic embedding spaces

We create two embedding spaces (old: 384-dim, new: 768-dim) that share a latent structure. This simulates what happens when two different embedding models encode the same texts — their vectors live in different dimensionalities but capture overlapping semantic content.

In [ ]:
rng = np.random.default_rng(42)

N = 2000  # number of calibration texts
D_OLD = 384  # old model dimension
D_NEW = 768  # new model dimension
LATENT = 64  # shared latent dimension

# Shared latent representation (what both models "see" about the text)
latent = rng.normal(size=(N, LATENT))

# Each model projects the latent representation into its own space
W_old = rng.normal(size=(LATENT, D_OLD)) / np.sqrt(LATENT)
W_new = rng.normal(size=(LATENT, D_NEW)) / np.sqrt(LATENT)

X_old = latent @ W_old  # (N, 384) — old model embeddings
Y_new = latent @ W_new  # (N, 768) — new model embeddings

# L2-normalize (as real embedding models do)
X_old = X_old / np.linalg.norm(X_old, axis=1, keepdims=True)
Y_new = Y_new / np.linalg.norm(Y_new, axis=1, keepdims=True)

print(f"Old model vectors: {X_old.shape}")
print(f"New model vectors: {Y_new.shape}")

## 2. Split into calibration and evaluation sets

The calibration set is what isotrieve uses to learn the mapping. The evaluation set is fresh data the mapping has never seen — this is what the quality gate checks.

In [ ]:
split = 1500
X_cal, X_eval = X_old[:split], X_old[split:]
Y_cal, Y_eval = Y_new[:split], Y_new[split:]

print(f"Calibration: {X_cal.shape[0]} pairs")
print(f"Evaluation:  {X_eval.shape[0]} pairs")

## 3. Fit the Ridge mapping

isotrieve learns a linear transformation `Y ≈ X W` (with optional bias term) using Ridge regression. Alpha is selected automatically via generalized cross-validation.

In [ ]:
mapping = RidgeMapping(alpha="auto", seed=0)
mapping.fit(X_cal, Y_cal)

report = mapping.validation_report()
print(f"Chosen alpha: {report.alpha:.4f}")
print(f"Train size: {report.n_train}, Holdout size: {report.n_holdout}")
print(f"Holdout cosine mean: {report.holdout_cosine_mean:.4f}")
print(f"Top-1 retention:     {report.top1_retention:.4f}")
print(f"Top-10 retention:    {report.top10_retention:.4f}")

## 4. Transform old vectors to new space

Now we can take any old-model vector and map it into the new model's space.

In [ ]:
# Transform a batch of old vectors to new space
X_mapped = mapping.transform(X_eval)

print(f"Input shape:  {X_eval.shape}  (old model space)")
print(f"Output shape: {X_mapped.shape}  (new model space)")

# How close are mapped vectors to the true new-model vectors?
cosine_sim = np.sum(X_mapped * Y_eval, axis=1)  # cosine since both are L2-normed
print("\nCosine similarity (mapped vs true new):")
print(f"  Mean:   {cosine_sim.mean():.4f}")
print(f"  Median: {np.median(cosine_sim):.4f}")
print(f"  P5:     {np.percentile(cosine_sim, 5):.4f}")

## 5. Run the quality gate

The quality gate predicts retrieval retention from proxy metrics (cosine similarity, rank correlation) without needing a full benchmark run. It returns PASS, WARN, or FAIL.

In [ ]:
gate = QualityGate()
gate_report = gate.evaluate(mapping, X_eval, Y_eval)

print(f"Verdict: {gate_report.verdict.value}")
print(f"Predicted retention: {gate_report.predicted_retention:.3f}")
print(f"Prediction interval: {gate_report.prediction_interval}")
print(f"Cosine mean:         {gate_report.cosine_mean:.4f}")
print(f"Top-1 retention:     {gate_report.top1_retention:.4f}")
print(f"Top-10 retention:    {gate_report.top10_retention:.4f}")
print(f"Holdout rank corr:   {gate_report.holdout_rank_corr:.4f}")
print(f"\nRationale: {gate_report.rationale}")

## 6. Save and load

The mapping is persisted as a single `.isotrieve` binary file containing the weight matrices and metadata.

In [ ]:
mapping.save("demo_mapping.isotrieve")

# Load it back
loaded = RidgeMapping.load("demo_mapping.isotrieve")
print(f"Loaded mapping: {loaded.d_src} -> {loaded.d_tgt}")
print(f"Fitted: {loaded.is_fitted}")

# Verify it produces the same output
X_loaded = loaded.transform(X_eval[:5])
print(f"Max diff from original: {np.max(np.abs(X_loaded - X_mapped[:5])):.2e}")

## What just happened

In plain English:

1. You had vectors embedded with an old model (384 dimensions).
2. You collected a small calibration set (~1500 texts) and embedded them with both old and new models.
3. isotrieve learned a linear transformation that maps old-space vectors into new-space vectors.
4. The quality gate checked that the mapping is good enough to use (PASS = safe to deploy).
5. You saved the mapping to a file. From now on, any old vector can be transformed to the new space instantly — no re-embedding needed.

**Key insight:** The mapping is tiny (a single matrix multiply), runs in microseconds per vector, and preserves retrieval quality. The calibration cost is O(K) embeddings instead of O(corpus_size) embeddings.

## Try it yourself

Change `LATENT` from 64 to 16 (making the two spaces more dissimilar). Does the gate verdict change? What about the cosine similarity?

```python
LATENT = 16  # try this
```